# cuDSS GPU test suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nardi/splineax/blob/main/notebooks/colab_gpu_tests.ipynb)

Runs `splineax`'s GPU-only test suite (the `CuDSS` solver) on a real CUDA GPU. This
solver cannot be exercised on ordinary CPU CI, since it wraps NVIDIA's CUDA-only cuDSS
library, so this notebook is the test path for it: clone a branch, install, and run
pytest, all against a real GPU.

> **Before running anything:** enable a GPU runtime first, *Runtime → Change runtime
> type → T4 GPU*. Every cell below depends on it.

In [ ]:
!nvidia-smi

## Preflight check

`splineax`'s `cudss` extra is marker-gated on `python_version >= "3.12"` (see
`pyproject.toml`). On an older Python, `pip install -e ".[cudss]"` further down
**succeeds and silently installs nothing**. Every GPU test then skips instead of
running, and the notebook looks like it passed when it did not test anything at all.
This cell turns that into an explicit, actionable failure up front, and does the same
for the CUDA driver version.

In [ ]:
import re
import subprocess
import sys

print(f"Python: {sys.version}")
python_ok = sys.version_info >= (3, 12)
print(f"  {'OK' if python_ok else 'FAIL'}: need Python >= 3.12.")

smi_output = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
match = re.search(r"CUDA Version:\s*(\d+)\.", smi_output)
cuda_major = int(match.group(1)) if match else None
print(f"CUDA driver major version: {cuda_major}")
cuda_ok = cuda_major is not None and cuda_major >= 13
print(f"  {'OK' if cuda_ok else 'FAIL'}: need a CUDA 13 driver.")

if python_ok and cuda_ok:
    print("\nPreflight passed: this runtime can run the cuDSS suite.")
else:
    raise RuntimeError(
        "Preflight failed (see above). Continuing past this point on a runtime "
        "that fails this check will not test cuDSS at all: on Python < 3.12 the "
        "install below succeeds but installs nothing, and on CUDA < 13 the install "
        "itself will fail. Pick a different runtime, or wait for Colab's image to "
        "catch up, before going further."
    )

## Clone, install, and restart

Set which fork and branch to test. Defaults to the branch this notebook shipped with.
Clones it, then installs `splineax` with its `cudss` extra. That extra pulls multi-GB
CUDA wheels (`jax[cuda13]`, cuDSS, cuSPARSE), so it takes a few minutes, and it
**upgrades Colab's preinstalled `jax`**, which is why the runtime has to be restarted
right after, before continuing to the sanity check below.

Two things to expect from the install, both fine:

- **`pip` prints a wall of dependency-conflict errors** about `torch`, `cudf`, `cuml`,
  and `pytensor` wanting `cuda-toolkit` 12.x or an older `numba`. Those are Colab's own
  preinstalled GPU packages complaining that this notebook moved the environment to
  CUDA 13. Nothing here uses them, and `pip` installs everything anyway.
- **JAX's CUDA 12 plugin gets uninstalled** right after. Colab ships one, cuDSS needs
  the CUDA 13 one, and leaving both in place makes them race to register a `"cuda"`
  backend. The loser prints a scary `ALREADY_EXISTS: PJRT_Api already exists for device
  type cuda` traceback, and if CUDA 12 is the one that wins the race, cuDSS cannot load
  at all.

In [ ]:
GITHUB_USERNAME = "nardi"  # @param {type:"string"}
BRANCH_NAME = "cudss"  # @param {type:"string"}

In [ ]:
!git clone --single-branch -b $BRANCH_NAME https://github.com/$GITHUB_USERNAME/splineax.git

In [ ]:
# `pytest-markdown-docs` is in the project's dev dependency group, not the `cudss`
# extra, but `pyproject.toml`'s `addopts` passes its flags to every pytest run. Without
# it installed, pytest exits with "unrecognized arguments: --markdown-docs".
!cd splineax && pip install -e ".[cudss]" pytest pytest-markdown-docs

# Colab preinstalls JAX's CUDA 12 PJRT plugin, and the install above adds the CUDA 13
# one (which cuDSS needs) without removing it. Both then register a "cuda" backend and
# one loses with an ALREADY_EXISTS error, so drop the one we do not want. Harmless if
# it was already absent: pip just skips it.
!pip uninstall -y jax-cuda12-plugin jax-cuda12-pjrt

### Restart the runtime now

*Runtime → Restart session*, then continue from the "Sanity check" cell below. The
cell right below this one does the restart for you, and Colab will prompt you to keep
going. Either way, execution has to resume on a fresh kernel from "Sanity check"
onward, since the install above replaced the preinstalled `jax`.

In [ ]:
import os

os.kill(os.getpid(), 9)

## Sanity check

In [ ]:
import os

# JAX grabs ~75% of the GPU on first use. This notebook then shells out to pytest,
# which starts a *second* JAX process that tries to do the same and spams
# "RESOURCE_EXHAUSTED: CUDA_ERROR_OUT_OF_MEMORY" while it backs off. Allocating on
# demand instead lets the kernel and the test runs share one GPU. Must be set before
# the first `import jax`, hence the placement at the top of this post-restart cell.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
import jax.numpy as jnp
import lineax as lx
from jax.experimental.sparse import BCOO

import splineax as splx

print("JAX devices:", jax.devices())
try:
    # Names the CUDA version JAX actually loaded, which is the thing to check first if
    # anything below fails.
    print("PJRT client:", jax.devices()[0].client.platform_version)
except Exception as exc:
    print("(could not read the PJRT client version:", exc, ")")

operator = splx.BCOOLinearOperator(BCOO.fromdense(jnp.eye(4) * 2.0 + 1.0))
chosen = splx.AutoSparseLinearSolver().select_solver(operator)
print("AutoSparseLinearSolver resolves to:", type(chosen).__name__)
assert isinstance(chosen, splx.CuDSS), (
    "expected CuDSS, got something else instead. The cudss extra likely did not "
    "really install (see the preflight cell above)."
)

# An actual solve, so a half-broken CUDA stack fails here with a clear message rather
# than several cells later inside pytest.
b = jnp.array([1.0, 2.0, 3.0, 4.0])
x = lx.linear_solve(operator, b, solver=chosen).value
residual = float(jnp.max(jnp.abs(operator.as_matrix() @ x - b)))
print(f"test solve residual: {residual:.2e}")
assert residual < 1e-4, "cuDSS solved, but got the wrong answer"
print("\nReady: cuDSS is installed, selected, and solving on this GPU.")

## Run the suite

An explicit path overrides `testpaths`, so this runs the solver suites without also
collecting the markdown-doc code fences. Those fences pin `splx.KLU()` explicitly, and
`KLU` is CPU-only, so they cannot pass on a GPU runtime.

For the same reason the `KLU` and `Pardiso` test modules skip themselves here, exactly
as the `CuDSS` ones skip on a CPU machine. What runs is everything backend-agnostic
plus the full `CuDSS` suite, which is the point of the exercise.

In [ ]:
!cd splineax && XLA_PYTHON_CLIENT_PREALLOCATE=false pytest tests/unit/solvers -v

Optionally, every unit test rather than just the solver suites:

In [ ]:
!cd splineax && XLA_PYTHON_CLIENT_PREALLOCATE=false pytest tests -v

## Reuse demo

"The tests passed" is one thing. Here is the factorization actually being reused.
Opens one `factorize_symbolic` scope and solves several different matrices sharing its
sparsity pattern, then checks cuDSS's own counters: the registry empties when the scope
closes, and the rebuild count (which only rises when a live factorization got evicted
and had to be silently rebuilt) never moves, because the whole demo comfortably fits in
the cache.

In [ ]:
import lineax as lx
from spineax import cudss

matrix = jnp.array(
    [
        [10.0, 2.0, 0.0, 0.0],
        [2.0, 8.0, 1.0, 0.0],
        [0.0, 1.0, 6.0, 3.0],
        [0.0, 0.0, 3.0, 5.0],
    ]
)
b = jnp.array([1.0, 2.0, 3.0, 4.0])
solver = splx.CuDSS()
rebuilds_before = cudss.rebuild_count()

with solver.factorize_symbolic(BCOO.fromdense(matrix)) as scope:
    for scale in [1.0, 2.0, 0.5, 3.0, 1.5]:
        scaled_operator = splx.BCOOLinearOperator(BCOO.fromdense(scale * matrix))
        state = scope.init(scaled_operator)
        x = lx.linear_solve(scaled_operator, b, solver=solver, state=state).value
        residual = float(jnp.max(jnp.abs(scale * matrix @ x - b)))
        print(f"scale={scale}: solved, residual={residual:.2e}")
    registry_size_while_open = cudss.registry_size()

registry_size_after_close = cudss.registry_size()
rebuilds_after = cudss.rebuild_count()

print(f"registry size while the scope was open: {registry_size_while_open}")
print(f"registry size after the scope closed:   {registry_size_after_close}")
print(f"rebuild count: {rebuilds_after - rebuilds_before} (0 means never evicted)")

assert registry_size_while_open >= 1
assert registry_size_after_close == 0, "the scope should have released its token"
assert rebuilds_after == rebuilds_before, (
    "the symbolic analysis was evicted and silently rebuilt \u2014 increase "
    "SPINEAX_FACTOR_CACHE if a real workload needs more headroom than this"
)
print("\nReuse confirmed: one analysis, five solves, no re-analysis.")